## imports

In [1]:
import os
import numpy as np
import tensorflow as tf
from keras_vit.vit import ViT_B16

# Minimize TensorFlow log messages
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

2025-04-25 20:28:26.853514: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-25 20:28:26.855330: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-25 20:28:26.864636: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-25 20:28:26.889473: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745600306.924152   36605 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745600306.93

## data

In [2]:
base_path = '../pistachio-edited'
train_path = f'{base_path}/train'
valid_path = f'{base_path}/valid'
test_path = f'{base_path}/test'

class_names = sorted([d.name for d in os.scandir(train_path) if d.is_dir()])
print("Class Names:", class_names)

Class Names: ['fungus', 'healthy', 'low-nutrition', 'pesil']


## model parameters

In [3]:
hp = {
    "image_size": 224,
    "num_channels": 3,
    "patch_size": 16,
    "num_patches": (128**2) // (25**2),
    "flat_patches_shape": ((128**2) // (25**2), 25*25*3),
    "batch_size": 32,
    "lr": 1e-4,
    "num_epochs": 50,
    "num_classes": len(class_names),
    "class_names": class_names,
    "num_layers": 12,
    "hidden_dim": 768,
    "mlp_dim": 3072,
    "num_heads": 12,
    "dropout_rate": 0.1
}
print(isinstance(hp['image_size'], int))

True


## load data

In [4]:
def load_data():
    train_gen = tf.keras.utils.image_dataset_from_directory(
        train_path,
        labels='inferred',
        batch_size=hp["batch_size"],
        image_size=(224,224),
        shuffle=True,
        seed=None,
        validation_split=None,
        subset=None,  # If you want to use validation split, set subset to 'training' or 'validation'
        interpolation='bilinear',
        follow_links=False
    )

    valid_gen = tf.keras.utils.image_dataset_from_directory(
        valid_path,
        labels='inferred',
        batch_size=hp["batch_size"],
        image_size=(224,224),
        shuffle=False,
        seed=None,
        validation_split=None,
        subset=None,
        interpolation='bilinear',
        follow_links=False
    )

    test_gen = tf.keras.utils.image_dataset_from_directory(
        test_path,
        labels='inferred',
        batch_size=hp["batch_size"],
        image_size=(224,224),
        shuffle=False,
        seed=None,
        validation_split=None,
        subset=None,
        interpolation='bilinear',
        follow_links=False
    )

    return train_gen, valid_gen, test_gen

# Load the datasets
train_x, valid_x, test_x = load_data()
print(f"Train: {len(train_x)} - Valid: {len(valid_x)} - Test: {len(test_x)}")

Found 2043 files belonging to 4 classes.
Found 205 files belonging to 4 classes.


2025-04-25 20:28:33.789688: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


Found 146 files belonging to 4 classes.
Train: 64 - Valid: 7 - Test: 5


## building model

In [5]:
model = ViT_B16(hp)
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(hp["lr"], clipvalue=1.0),
    metrics=["accuracy"]
)

model.summary()

# Set up callbacks
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=10, min_lr=1e-10, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)
]

ValueError: Invalid dtype: TrackedDict